# Day 4: Applications of Geometric Deep Learning

**Geometric Deep Learning Workshop**

---

## Learning Objectives

1. Convert molecular representations (SMILES) to graph structures
2. Build and train a molecular property prediction model with PyG
3. Apply GNNs to social network analysis and community detection
4. Complete a capstone project template integrating the week's concepts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GINConv, global_mean_pool, global_add_pool
from torch_geometric.datasets import TUDataset
from torch_geometric.utils import to_networkx
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score

np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch: {torch.__version__}")

---

## 1. Molecular Graphs: From SMILES to Graph Representations

Molecules are naturally represented as graphs:
- **Nodes** = atoms (with features like atomic number, charge, hybridization)
- **Edges** = chemical bonds (with features like bond type, aromaticity)

**SMILES** (Simplified Molecular Input Line Entry System) is a string notation for molecules:
- `C` = carbon, `O` = oxygen, `N` = nitrogen
- `=` = double bond, `#` = triple bond
- `()` = branching, numbers = ring closures

Examples:
- Water: `O`
- Ethanol: `CCO`
- Benzene: `c1ccccc1`
- Aspirin: `CC(=O)Oc1ccccc1C(=O)O`

In [ ]:
# Manual SMILES-to-graph conversion (simplified, for educational purposes)
# In practice, use RDKit: from rdkit import Chem

# We will manually define a few small molecules as graphs

# Atom type encoding (one-hot)
ATOM_TYPES = {'C': 0, 'N': 1, 'O': 2, 'S': 3, 'F': 4, 'Cl': 5, 'Br': 6, 'H': 7}
NUM_ATOM_TYPES = len(ATOM_TYPES)


def atom_one_hot(atom_symbol):
    """One-hot encode an atom type."""
    vec = [0.0] * NUM_ATOM_TYPES
    if atom_symbol in ATOM_TYPES:
        vec[ATOM_TYPES[atom_symbol]] = 1.0
    return vec


def create_molecule_graph(atoms, bonds, label=None):
    """Create a PyG Data object for a molecule.

    Args:
        atoms: list of atom symbols, e.g., ['C', 'C', 'O']
        bonds: list of (i, j) tuples for bonds
        label: integer class label
    """
    # Node features: one-hot atom type
    x = torch.tensor([atom_one_hot(a) for a in atoms], dtype=torch.float)

    # Edge index (undirected: add both directions)
    src = [b[0] for b in bonds] + [b[1] for b in bonds]
    dst = [b[1] for b in bonds] + [b[0] for b in bonds]
    edge_index = torch.tensor([src, dst], dtype=torch.long)

    data = Data(x=x, edge_index=edge_index)
    if label is not None:
        data.y = torch.tensor([label], dtype=torch.long)

    return data


# Create some example molecules
molecules = {
    'Methane (CH4)': {
        'atoms': ['C', 'H', 'H', 'H', 'H'],
        'bonds': [(0,1), (0,2), (0,3), (0,4)],
        'smiles': 'C'
    },
    'Ethanol (C2H5OH)': {
        'atoms': ['C', 'C', 'O', 'H', 'H', 'H', 'H', 'H', 'H'],
        'bonds': [(0,1), (1,2), (0,3), (0,4), (0,5), (1,6), (1,7), (2,8)],
        'smiles': 'CCO'
    },
    'Benzene (C6H6)': {
        'atoms': ['C', 'C', 'C', 'C', 'C', 'C'],
        'bonds': [(0,1), (1,2), (2,3), (3,4), (4,5), (5,0)],
        'smiles': 'c1ccccc1'
    },
    'Aniline (C6H5NH2)': {
        'atoms': ['C', 'C', 'C', 'C', 'C', 'C', 'N'],
        'bonds': [(0,1), (1,2), (2,3), (3,4), (4,5), (5,0), (0,6)],
        'smiles': 'c1ccccc1N'
    },
}

# Visualize molecular graphs
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

atom_colors = {'C': 'gray', 'N': 'steelblue', 'O': 'coral', 'H': 'lightgreen',
               'S': 'gold', 'F': 'cyan', 'Cl': 'lime', 'Br': 'brown'}

for idx, (name, mol_info) in enumerate(molecules.items()):
    ax = axes[idx]
    data = create_molecule_graph(mol_info['atoms'], mol_info['bonds'])
    G = to_networkx(data, to_undirected=True)

    colors = [atom_colors.get(a, 'white') for a in mol_info['atoms']]
    labels = {i: mol_info['atoms'][i] for i in range(len(mol_info['atoms']))}

    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, ax=ax, node_color=colors, labels=labels,
            node_size=400, font_size=10, font_weight='bold',
            edge_color='gray', width=2)
    ax.set_title(f"{name}\nSMILES: {mol_info['smiles']}", fontsize=10)

plt.tight_layout()
plt.show()

---

## 2. Molecular Property Prediction with PyG

We will train a GNN to predict molecular properties using the **MUTAG** dataset:
- 188 chemical compounds
- Binary classification: mutagenic or non-mutagenic
- Node features: atom type (one-hot, 7 types)
- Edge features: bond type

In [ ]:
# Load MUTAG dataset
dataset = TUDataset(root='/tmp/MUTAG', name='MUTAG')

print(f"Dataset: {dataset}")
print(f"Number of graphs: {len(dataset)}")
print(f"Number of node features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

# Dataset statistics
num_nodes = [d.num_nodes for d in dataset]
num_edges = [d.num_edges for d in dataset]
labels = [d.y.item() for d in dataset]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(num_nodes, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Number of Nodes')
axes[0].set_ylabel('Count')
axes[0].set_title('Node Count Distribution')
axes[0].grid(True, alpha=0.3)

axes[1].hist(num_edges, bins=20, color='coral', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Number of Edges')
axes[1].set_ylabel('Count')
axes[1].set_title('Edge Count Distribution')
axes[1].grid(True, alpha=0.3)

axes[2].bar(['Non-mutagenic (0)', 'Mutagenic (1)'],
            [labels.count(0), labels.count(1)],
            color=['steelblue', 'coral'], edgecolor='black')
axes[2].set_ylabel('Count')
axes[2].set_title('Class Distribution')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAvg nodes: {np.mean(num_nodes):.1f}, Avg edges: {np.mean(num_edges):.1f}")

In [ ]:
# GIN (Graph Isomorphism Network) - more expressive than GCN for graph classification
# GIN uses: h_v^{(k)} = MLP^{(k)}((1 + eps) * h_v^{(k-1)} + SUM(h_u^{(k-1)}))

class MoleculeGIN(nn.Module):
    """Graph Isomorphism Network for molecular property prediction."""

    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=4):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        for i in range(num_layers):
            in_dim = in_channels if i == 0 else hidden_channels
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden_channels),
                nn.BatchNorm1d(hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
            )
            self.convs.append(GINConv(mlp, train_eps=True))
            self.bns.append(nn.BatchNorm1d(hidden_channels))

        # Graph-level readout + classifier
        self.fc1 = nn.Linear(hidden_channels, hidden_channels)
        self.fc2 = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch):
        # GNN layers
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=0.3, training=self.training)

        # Graph-level pooling
        x = global_add_pool(x, batch)

        # Classifier
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.fc2(x)
        return x

    def get_graph_embeddings(self, x, edge_index, batch):
        """Get graph-level embeddings before classification."""
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
        return global_add_pool(x, batch)


model = MoleculeGIN(
    in_channels=dataset.num_features,
    hidden_channels=64,
    out_channels=dataset.num_classes,
    num_layers=4,
)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Train/test split with cross-validation style
torch.manual_seed(42)
dataset = dataset.shuffle()

n_train = int(0.8 * len(dataset))
train_dataset = dataset[:n_train]
test_dataset = dataset[n_train:]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
all_loader = DataLoader(dataset, batch_size=64, shuffle=False)

# Training
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
criterion = nn.CrossEntropyLoss()

train_losses = []
train_accs = []
test_accs = []

for epoch in range(150):
    # Train
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs
        correct += (out.argmax(1) == batch.y).sum().item()
        total += batch.num_graphs

    scheduler.step()

    # Evaluate
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for batch in test_loader:
            out = model(batch.x, batch.edge_index, batch.batch)
            test_correct += (out.argmax(1) == batch.y).sum().item()
            test_total += batch.num_graphs

    train_losses.append(total_loss / total)
    train_accs.append(correct / total)
    test_accs.append(test_correct / test_total)

    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {train_losses[-1]:.4f} | "
              f"Train: {train_accs[-1]:.4f} | Test: {test_accs[-1]:.4f}")

print(f"\nBest test accuracy: {max(test_accs):.4f} (epoch {np.argmax(test_accs)+1})")

In [ ]:
# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, color='steelblue')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss (MUTAG)')
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, label='Train', color='steelblue')
ax2.plot(test_accs, label='Test', color='coral')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Molecular Property Prediction (MUTAG)')
ax2.legend()
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize learned graph embeddings
model.eval()
all_embeddings = []
all_labels = []

with torch.no_grad():
    for batch in all_loader:
        emb = model.get_graph_embeddings(batch.x, batch.edge_index, batch.batch)
        all_embeddings.append(emb.numpy())
        all_labels.append(batch.y.numpy())

embeddings = np.concatenate(all_embeddings, axis=0)
labels_arr = np.concatenate(all_labels, axis=0)

# t-SNE visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
emb_2d = tsne.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(8, 6))
for label, name, color in [(0, 'Non-mutagenic', 'steelblue'), (1, 'Mutagenic', 'coral')]:
    mask = labels_arr == label
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], c=color, label=name,
              s=40, alpha=0.7, edgecolors='black', linewidths=0.3)

ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
ax.set_title('GIN Graph Embeddings of MUTAG Molecules', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 3. Social Network Analysis with GNN Embeddings

GNNs learn node embeddings that capture structural roles in a network.
We can use these embeddings for **community detection** by clustering in
the embedding space.

In [ ]:
# Generate a synthetic social network with community structure

def generate_community_graph(sizes, p_intra=0.3, p_inter=0.01, seed=42):
    """Generate a stochastic block model graph.

    Args:
        sizes: list of community sizes
        p_intra: probability of edges within communities
        p_inter: probability of edges between communities
    """
    n_communities = len(sizes)
    probs = np.full((n_communities, n_communities), p_inter)
    np.fill_diagonal(probs, p_intra)
    G = nx.stochastic_block_model(sizes, probs.tolist(), seed=seed)

    # Ground truth labels
    labels = []
    for i, size in enumerate(sizes):
        labels.extend([i] * size)

    return G, labels


# Create a graph with 4 communities
community_sizes = [30, 25, 35, 20]
G_social, true_labels = generate_community_graph(
    community_sizes, p_intra=0.25, p_inter=0.01
)

print(f"Social network: {G_social.number_of_nodes()} nodes, {G_social.number_of_edges()} edges")
print(f"Communities: {len(community_sizes)} (sizes: {community_sizes})")

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
pos = nx.spring_layout(G_social, seed=42, k=0.3)
nx.draw(G_social, pos, ax=ax, node_color=true_labels, cmap='Set2',
        node_size=80, edge_color='lightgray', width=0.3, alpha=0.9)
ax.set_title('Synthetic Social Network (colored by true communities)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Convert to PyG and train a GNN for unsupervised community detection
# Strategy: use a GNN autoencoder to learn node embeddings,
# then cluster the embeddings

from torch_geometric.utils import from_networkx

# Convert NetworkX graph to PyG
# Assign one-hot degree features as node attributes
max_degree = max(dict(G_social.degree()).values())
for node in G_social.nodes():
    degree = G_social.degree(node)
    feat = [0.0] * (max_degree + 1)
    feat[degree] = 1.0
    G_social.nodes[node]['x'] = feat

data_social = from_networkx(G_social, group_node_attrs=['x'])
data_social.y = torch.tensor(true_labels, dtype=torch.long)

print(f"PyG Data: {data_social}")
print(f"Node features: {data_social.x.shape}")

In [ ]:
class GNNEncoder(nn.Module):
    """GNN encoder that learns node embeddings for community detection."""

    def __init__(self, in_channels, hidden_channels, emb_dim):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, emb_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        return x


class LinkPredictionDecoder(nn.Module):
    """Decode edges from node embeddings using inner product."""

    def forward(self, z, edge_index):
        # Inner product between source and target embeddings
        src, dst = edge_index
        return (z[src] * z[dst]).sum(dim=1)

    def full_decode(self, z):
        # Full adjacency reconstruction
        return torch.sigmoid(z @ z.T)


class GraphAutoEncoder(nn.Module):
    """Graph Autoencoder: encode nodes, decode edges."""

    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def encode(self, x, edge_index):
        return self.encoder(x, edge_index)

    def decode(self, z, edge_index):
        return self.decoder(z, edge_index)


# Initialize
emb_dim = 16
encoder = GNNEncoder(data_social.num_features, 32, emb_dim)
decoder = LinkPredictionDecoder()
gae = GraphAutoEncoder(encoder, decoder)

print(gae)

In [ ]:
# Train the Graph Autoencoder
# Loss: reconstruct edges (positive edges should have high scores,
#        negative/non-edges should have low scores)

optimizer = torch.optim.Adam(gae.parameters(), lr=0.01, weight_decay=1e-5)

def negative_sampling(edge_index, num_nodes, num_neg_samples):
    """Sample random non-edges as negative examples."""
    neg_src = torch.randint(0, num_nodes, (num_neg_samples,))
    neg_dst = torch.randint(0, num_nodes, (num_neg_samples,))
    return torch.stack([neg_src, neg_dst])


losses = []

for epoch in range(200):
    gae.train()
    optimizer.zero_grad()

    # Encode
    z = gae.encode(data_social.x, data_social.edge_index)

    # Positive edge scores
    pos_scores = gae.decode(z, data_social.edge_index)
    pos_loss = F.binary_cross_entropy_with_logits(
        pos_scores, torch.ones_like(pos_scores)
    )

    # Negative edge scores
    neg_edge_index = negative_sampling(
        data_social.edge_index, data_social.num_nodes,
        num_neg_samples=data_social.edge_index.size(1)
    )
    neg_scores = gae.decode(z, neg_edge_index)
    neg_loss = F.binary_cross_entropy_with_logits(
        neg_scores, torch.zeros_like(neg_scores)
    )

    loss = pos_loss + neg_loss
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

In [ ]:
# Extract embeddings and cluster
gae.eval()
with torch.no_grad():
    z = gae.encode(data_social.x, data_social.edge_index).numpy()

# K-Means clustering on embeddings
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
predicted_communities = kmeans.fit_predict(z)

# Evaluate using Normalized Mutual Information
nmi = normalized_mutual_info_score(true_labels, predicted_communities)
print(f"NMI (GNN embeddings): {nmi:.4f}")

# Compare with spectral clustering baseline
from sklearn.cluster import SpectralClustering
adj = nx.adjacency_matrix(G_social).toarray()
spec_clustering = SpectralClustering(n_clusters=4, affinity='precomputed',
                                     random_state=42)
spec_labels = spec_clustering.fit_predict(adj)
nmi_spec = normalized_mutual_info_score(true_labels, spec_labels)
print(f"NMI (spectral clustering): {nmi_spec:.4f}")

In [ ]:
# Visualize the results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Ground truth
nx.draw(G_social, pos, ax=axes[0], node_color=true_labels, cmap='Set2',
        node_size=80, edge_color='lightgray', width=0.3)
axes[0].set_title('Ground Truth Communities', fontsize=12)

# GNN predictions
nx.draw(G_social, pos, ax=axes[1], node_color=predicted_communities, cmap='Set2',
        node_size=80, edge_color='lightgray', width=0.3)
axes[1].set_title(f'GNN Embedding + K-Means (NMI={nmi:.3f})', fontsize=12)

# Spectral clustering
nx.draw(G_social, pos, ax=axes[2], node_color=spec_labels, cmap='Set2',
        node_size=80, edge_color='lightgray', width=0.3)
axes[2].set_title(f'Spectral Clustering (NMI={nmi_spec:.3f})', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Embedding space visualization
tsne_social = TSNE(n_components=2, random_state=42, perplexity=20)
emb_2d_social = tsne_social.fit_transform(z)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Colored by ground truth
scatter1 = ax1.scatter(emb_2d_social[:, 0], emb_2d_social[:, 1],
                       c=true_labels, cmap='Set2', s=40,
                       edgecolors='black', linewidths=0.3)
ax1.set_title('GNN Embeddings (colored by ground truth)', fontsize=12)
ax1.set_xlabel('t-SNE dim 1')
ax1.set_ylabel('t-SNE dim 2')
ax1.grid(True, alpha=0.3)

# Colored by K-Means predictions
scatter2 = ax2.scatter(emb_2d_social[:, 0], emb_2d_social[:, 1],
                       c=predicted_communities, cmap='Set2', s=40,
                       edgecolors='black', linewidths=0.3)
ax2.set_title('GNN Embeddings (colored by K-Means)', fontsize=12)
ax2.set_xlabel('t-SNE dim 1')
ax2.set_ylabel('t-SNE dim 2')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 4. Capstone Project Template

This template provides a structured starting point for your own GDL project.
Choose one of the suggested directions or define your own.

### Suggested Project Directions

1. **Molecular property prediction**: Use PyG's QM9 or ZINC datasets. Predict a continuous property (regression) with SchNet or DimeNet.

2. **Protein structure analysis**: Represent protein backbones as graphs (residues = nodes, contacts = edges). Classify protein families.

3. **Traffic flow prediction**: Model a road network as a graph. Predict future traffic from historical data using spatio-temporal GNNs.

4. **Point cloud segmentation**: Extend PointNet to per-point classification. Use ShapeNet or S3DIS datasets.

5. **Knowledge graph completion**: Use relational GNNs (R-GCN) to predict missing links in knowledge graphs.

In [ ]:
# ============================================================
# CAPSTONE PROJECT TEMPLATE
# ============================================================

# ------ Step 1: Define your problem ------
PROJECT_NAME = "My GDL Project"  # <-- Change this
TASK_TYPE = "classification"      # classification / regression / generation

print(f"Project: {PROJECT_NAME}")
print(f"Task: {TASK_TYPE}")

In [ ]:
# ------ Step 2: Load or create your dataset ------

# Option A: Use a PyG built-in dataset
# from torch_geometric.datasets import Planetoid, TUDataset, QM9, ShapeNet
# dataset = TUDataset(root='/tmp/PROTEINS', name='PROTEINS')

# Option B: Create from custom data
# from torch_geometric.data import Data, InMemoryDataset

# Example: loading PROTEINS dataset
from torch_geometric.datasets import TUDataset

dataset = TUDataset(root='/tmp/PROTEINS', name='PROTEINS')

print(f"Dataset: {dataset}")
print(f"Graphs: {len(dataset)}")
print(f"Features: {dataset.num_features}")
print(f"Classes: {dataset.num_classes}")

# Analyze the dataset
print(f"\n--- Dataset Statistics ---")
sizes = [d.num_nodes for d in dataset]
print(f"Avg nodes: {np.mean(sizes):.1f} +/- {np.std(sizes):.1f}")
print(f"Min/Max nodes: {min(sizes)}/{max(sizes)}")

In [ ]:
# ------ Step 3: Define your model ------

class CapstoneGNN(nn.Module):
    """Template GNN model. Customize for your task."""

    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_layers=3, dropout=0.3, task='graph'):
        super().__init__()
        self.task = task  # 'graph' or 'node'
        self.dropout = dropout

        # GNN backbone
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        self.convs.append(GCNConv(in_channels, hidden_channels))
        self.bns.append(nn.BatchNorm1d(hidden_channels))

        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(nn.BatchNorm1d(hidden_channels))

        # Task-specific head
        self.head = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, out_channels),
        )

    def forward(self, x, edge_index, batch=None):
        # GNN layers with residual connections
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            x_new = conv(x, edge_index)
            x_new = bn(x_new)
            x_new = F.relu(x_new)
            x_new = F.dropout(x_new, p=self.dropout, training=self.training)
            # Residual connection (if dimensions match)
            if i > 0:
                x_new = x_new + x
            x = x_new

        # Pooling for graph-level tasks
        if self.task == 'graph' and batch is not None:
            x = global_mean_pool(x, batch)

        # Task head
        return self.head(x)


model = CapstoneGNN(
    in_channels=dataset.num_features,
    hidden_channels=64,
    out_channels=dataset.num_classes,
    num_layers=3,
    task='graph',
)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ------ Step 4: Training pipeline ------

# Split data
torch.manual_seed(42)
dataset = dataset.shuffle()
n_train = int(0.8 * len(dataset))
n_val = int(0.1 * len(dataset))

train_set = dataset[:n_train]
val_set = dataset[n_train:n_train+n_val]
test_set = dataset[n_train+n_val:]

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=20
)
criterion = nn.CrossEntropyLoss()


def train_one_epoch():
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
        correct += (out.argmax(1) == batch.y).sum().item()
        total += batch.num_graphs
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    for batch in loader:
        out = model(batch.x, batch.edge_index, batch.batch)
        correct += (out.argmax(1) == batch.y).sum().item()
        total += batch.num_graphs
    return correct / total


# Training loop with early stopping
best_val_acc = 0
patience_counter = 0
max_patience = 30
history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'test_acc': []}

for epoch in range(150):
    train_loss, train_acc = train_one_epoch()
    val_acc = evaluate(val_loader)
    test_acc = evaluate(test_loader)

    scheduler.step(train_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['test_acc'].append(test_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_test_acc = test_acc
        patience_counter = 0
        # Save best model
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {train_loss:.4f} | "
              f"Train: {train_acc:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")

    if patience_counter >= max_patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nBest validation accuracy: {best_val_acc:.4f}")
print(f"Corresponding test accuracy: {best_test_acc:.4f}")

In [ ]:
# ------ Step 5: Analyze results ------

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], color='steelblue')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train', color='steelblue')
ax2.plot(history['val_acc'], label='Validation', color='coral')
ax2.plot(history['test_acc'], label='Test', color='seagreen', alpha=0.7)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title(f'Capstone: {PROJECT_NAME}')
ax2.legend()
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ------ Step 6: Report your findings ------

print("=" * 60)
print(f"PROJECT REPORT: {PROJECT_NAME}")
print("=" * 60)
print(f"\nDataset: {dataset}")
print(f"  Graphs: {len(dataset)}, Features: {dataset.num_features}, Classes: {dataset.num_classes}")
print(f"\nModel: CapstoneGNN")
print(f"  Layers: {len(model.convs)}, Hidden dim: 64, Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nResults:")
print(f"  Best Validation Accuracy: {best_val_acc:.4f}")
print(f"  Corresponding Test Accuracy: {best_test_acc:.4f}")
print(f"\nNext Steps:")
print("  - [ ] Try different GNN architectures (GAT, GIN, GraphSAGE)")
print("  - [ ] Experiment with different pooling (sum, max, attention)")
print("  - [ ] Add edge features if available")
print("  - [ ] Hyperparameter search (learning rate, hidden dim, depth)")
print("  - [ ] Cross-validation for robust evaluation")
print("  - [ ] Visualize learned embeddings")
print("  - [ ] Analyze failure cases")

---

## Workshop Summary

| Day | Topics | Key Models |
|-----|--------|------------|
| 1 | Graph representations, Message Passing | GCN |
| 2 | Spectral methods, Attention, Graph classification | GAT, Oversmoothing |
| 3 | Point clouds, Equivariance, TDA | PointNet, SE(3) invariance |
| 4 | Molecular graphs, Social networks, Capstone | GIN, Graph AutoEncoder |

### Recommended Resources

- **Textbook**: *Geometric Deep Learning: Grids, Groups, Graphs, Geodesics, and Gauges* (Bronstein et al., 2021)
- **Course**: Stanford CS224W: Machine Learning with Graphs
- **Library docs**: [PyTorch Geometric](https://pytorch-geometric.readthedocs.io/)
- **Papers**: Kipf & Welling (2017), Velickovic et al. (2018), Qi et al. (2017), Gilmer et al. (2017)

---

### Exercises

1. **RDKit integration**: If you have RDKit installed, write a function to convert a SMILES string to a PyG Data object with proper atom/bond features.
2. **GIN vs GCN**: Compare GIN and GCN on MUTAG. Which is better and why?
3. **Larger social network**: Generate a network with 1000 nodes and 8 communities. How does the GNN scale?
4. **Capstone**: Complete the project template with your own dataset and model. Aim for a short report.